# Codebase RAG with SST-Tree Code Splitting

This notebook implements a **Retrieval-Augmented Generation** system for querying a codebase using natural language. It uses a **Syntax-/Structure-aware Tree (SST)** splitting strategy that respects the syntactic boundaries of code rather than naive character splitting.

## How it works

1. **Scan** a folder of source code recursively
2. **Split** each file into semantically meaningful chunks using SST (functions, classes, methods, blocks)
3. **Embed** each chunk with Ollama (`nomic-embed-text`)
4. **Index** chunks in ChromaDB with rich metadata (file, line range, chunk type, docstring)
5. **Query** with natural language — retrieve relevant chunks and answer with an Ollama LLM

## SST-Tree Splitting Strategy

Instead of splitting code at arbitrary character boundaries (which can break syntax mid-function), SST-tree splitting:
- For **Python**: uses the built-in `ast` module to find functions, classes, methods, and top-level blocks
- For **other languages** (JS, TS, Java, etc.): uses indentation-aware structural parsing with heuristic detection of functions, classes, and control flow
- Preserves **docstrings**, **signatures**, and **line numbers** in metadata
- Each chunk is a complete, semantically coherent unit of code

In [1]:
# @install: pip install chromadb langchain-ollama ollama

from __future__ import annotations

import ast
import os
import re
import textwrap
import json
import time
from pathlib import Path
from typing import Generator, Optional
from dataclasses import dataclass, field

import chromadb
from chromadb.config import Settings as ChromaSettings
from langchain_ollama import OllamaEmbeddings, ChatOllama

print("All imports OK")

Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


All imports OK


## 1. SST-Tree Code Splitter

The splitter parses each source file and breaks it into semantically coherent chunks.

### Python splitting (via `ast` module)
- Identifies **classes**, **functions**, **methods**, **module-level assignments**
- Preserves docstrings and signatures
- Each node becomes one chunk with metadata

### Generic code splitting (indentation-aware)
- Detects blocks by tracking indentation changes
- Identifies function/class definitions via regex
- Groups consecutive single-line statements into context chunks

### Fallback
- Files that can't be parsed structurally are split by blank-line-separated sections

In [2]:
@dataclass
class CodeChunk:
    """A single semantically-coherent chunk of code."""
    file_path: str
    language: str
    chunk_type: str  # 'function', 'class', 'method', 'module_block', 'imports', 'config'
    name: str = ''
    start_line: int = 0
    end_line: int = 0
    docstring: str = ''
    content: str = ''
    signature: str = ''  # e.g. 'def foo(x: int) -> str:'
    parent_class: str = ''  # if chunk is a method
    relative_path: str = ''

    def to_dict(self) -> dict:
        return {
            'file_path': self.file_path,
            'relative_path': self.relative_path,
            'language': self.language,
            'chunk_type': self.chunk_type,
            'name': self.name,
            'start_line': self.start_line,
            'end_line': self.end_line,
            'docstring': self.docstring[:500] if self.docstring else '',
            'signature': self.signature,
            'parent_class': self.parent_class,
        }


LANGUAGE_MAP: dict[str, str] = {
    '.py': 'python',
    '.js': 'javascript',
    '.ts': 'typescript',
    '.jsx': 'javascriptreact',
    '.tsx': 'typescriptreact',
    '.java': 'java',
    '.c': 'c',
    '.cpp': 'cpp',
    '.h': 'c',
    '.hpp': 'cpp',
    '.go': 'go',
    '.rs': 'rust',
    '.rb': 'ruby',
    '.sh': 'bash',
    '.R': 'r',
    '.ipynb': 'jupyter',
    '.md': 'markdown',
    '.yaml': 'yaml',
    '.yml': 'yaml',
    '.json': 'json',
    '.toml': 'toml',
    '.cfg': 'ini',
    '.ini': 'ini',
}


class SSTCodeSplitter:
    """Syntax-Structure Tree (SST) code splitter.
    
    Splits code into semantically meaningful chunks by parsing the structure
    of the code rather than splitting at arbitrary character boundaries.
    """
    
    def __init__(
        self,
        max_chunk_lines: int = 200,
        min_chunk_lines: int = 3,
        include_comments: bool = True,
        include_docstrings: bool = True,
    ):
        self.max_chunk_lines = max_chunk_lines
        self.min_chunk_lines = min_chunk_lines
        self.include_comments = include_comments
        self.include_docstrings = include_docstrings
    
    def split_file(self, file_path: str) -> list[CodeChunk]:
        """Split a single file into chunks using SST."""
        ext = Path(file_path).suffix.lower()
        language = LANGUAGE_MAP.get(ext, 'unknown')
        
        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            source = f.read()
        
        if not source.strip():
            return []
        
        lines = source.split('\n')
        
        if language == 'python':
            chunks = self._split_python(source, file_path, lines, language)
        elif language in ('javascript', 'typescript', 'javascriptreact', 'typescriptreact',
                          'java', 'c', 'cpp', 'go', 'rust', 'ruby'):
            chunks = self._split_braced_language(source, file_path, lines, language)
        else:
            chunks = self._split_fallback(source, file_path, lines, language)
        
        # Filter small chunks and set relative paths
        result = []
        for c in chunks:
            if c.end_line - c.start_line < self.min_chunk_lines and c.chunk_type not in ('module_block', 'imports', 'config'):
                continue
            c.relative_path = self._get_relative_path(c.file_path)
            result.append(c)
        
        return result
    
    # ----- Python (ast-based) -----
    
    def _split_python(self, source: str, file_path: str, lines: list[str], language: str) -> list[CodeChunk]:
        chunks = []
        try:
            tree = ast.parse(source)
        except SyntaxError:
            return self._split_fallback(source, file_path, lines, language)
        
        for node in ast.iter_child_nodes(tree):
            child_chunks = self._python_node_to_chunks(node, source, file_path, lines, language)
            chunks.extend(child_chunks)
        
        # If no structure found, fall back
        if not chunks:
            return self._split_fallback(source, file_path, lines, language)
        
        return chunks
    
    def _python_node_to_chunks(
        self, node: ast.AST, source: str, file_path: str,
        lines: list[str], language: str, parent_class: str = ''
    ) -> list[CodeChunk]:
        chunks = []
        start_line = getattr(node, 'lineno', 1) - 1  # 0-indexed
        end_line = getattr(node, 'end_lineno', start_line + 1)
        
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            chunk = CodeChunk(
                file_path=file_path,
                language=language,
                chunk_type='method' if parent_class else 'function',
                name=node.name,
                start_line=start_line + 1,
                end_line=end_line,
                parent_class=parent_class,
                content='\n'.join(lines[start_line:end_line]),
            )
            # Extract signature (first line of the function)
            if start_line < len(lines):
                sig_lines = []
                for i in range(start_line, min(end_line, len(lines))):
                    sig_lines.append(lines[i])
                    if lines[i].rstrip().endswith(':'):
                        break
                chunk.signature = '\n'.join(sig_lines)
            # Extract docstring
            docstring = ast.get_docstring(node)
            if docstring:
                chunk.docstring = docstring
            chunks.append(chunk)
        
        elif isinstance(node, ast.ClassDef):
            # Create a chunk for the class itself (header + docstring)
            class_end_line = start_line + 1
            for child in ast.iter_child_nodes(node):
                child_end = getattr(child, 'end_lineno', start_line + 1)
                if isinstance(child, ast.Expr) and hasattr(child, 'value'):
                    if isinstance(child.value, ast.Constant) and isinstance(child.value.value, str):
                        class_end_line = child_end
            
            # Class header chunk
            chunk = CodeChunk(
                file_path=file_path,
                language=language,
                chunk_type='class',
                name=node.name,
                start_line=start_line + 1,
                end_line=class_end_line,
                content='\n'.join(lines[start_line:class_end_line]),
            )
            docstring = ast.get_docstring(node)
            if docstring:
                chunk.docstring = docstring
            chunks.append(chunk)
            
            # Recursively extract methods
            for child in ast.iter_child_nodes(node):
                child_chunks = self._python_node_to_chunks(
                    child, source, file_path, lines, language, parent_class=node.name
                )
                chunks.extend(child_chunks)
        
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            # Group imports
            chunk = CodeChunk(
                file_path=file_path,
                language=language,
                chunk_type='imports',
                start_line=start_line + 1,
                end_line=end_line,
                content='\n'.join(lines[start_line:end_line]),
            )
            chunks.append(chunk)
        
        elif isinstance(node, ast.Assign) and len(node.targets) == 1:
            if isinstance(node.targets[0], ast.Name) and node.targets[0].id.isupper():
                chunk = CodeChunk(
                    file_path=file_path,
                    language=language,
                    chunk_type='config',
                    name=node.targets[0].id,
                    start_line=start_line + 1,
                    end_line=end_line,
                    content='\n'.join(lines[start_line:end_line]),
                )
                chunks.append(chunk)
        
        # Skip other node types (they'll be part of parent chunks)
        return chunks
    
    # ----- Braced languages (indentation + brace matching) -----
    
    def _split_braced_language(self, source: str, file_path: str,
                                lines: list[str], language: str) -> list[CodeChunk]:
        chunks = []
        i = 0
        while i < len(lines):
            line = lines[i]
            stripped = line.strip()
            
            # Detect function/class/method definition
            func_match = re.match(
                r'^(?:export\s+)?(?:async\s+)?function\s+\*?\s*(\w+)\s*\(',
                stripped
            )
            class_match = re.match(r'^(?:export\s+)?class\s+(\w+)', stripped)
            method_match = re.match(r'^(?:async\s+)?(\w+)\s*\(', stripped) if not func_match else None
            arrow_func = re.match(r'^(?:export\s+)?(?:const|let|var)\s+(\w+)\s*=\s*(?:\([^)]*\)|\w+)\s*=>', stripped)
            
            if func_match or class_match or arrow_func:
                chunk_type = 'class' if class_match else 'function'
                name = (class_match or func_match or arrow_func).group(1)
                start = i
                end = self._find_brace_block_end(lines, i)
                
                chunk = CodeChunk(
                    file_path=file_path,
                    language=language,
                    chunk_type=chunk_type,
                    name=name,
                    start_line=start + 1,
                    end_line=end + 1,
                    content='\n'.join(lines[start:end + 1]),
                    signature=stripped,
                )
                chunks.append(chunk)
                i = end + 1
                continue
            
            i += 1
        
        if not chunks:
            return self._split_fallback(source, file_path, lines, language)
        
        return chunks
    
    def _find_brace_block_end(self, lines: list[str], start: int) -> int:
        """Find the matching closing brace for a block starting at `start`."""
        brace_count = 0
        started = False
        for i in range(start, len(lines)):
            line = lines[i]
            if not started and '{' not in line:
                # Skip signature lines
                for ch in line:
                    if ch in '({[':
                        brace_count += 1
                        started = True
                continue
            started = True
            for ch in line:
                if ch in '({[':
                    brace_count += 1
                elif ch in ')}]':
                    brace_count -= 1
            if brace_count <= 0 and started:
                return i
        return len(lines) - 1
    
    # ----- Fallback (blank-line sections) -----
    
    def _split_fallback(self, source: str, file_path: str,
                         lines: list[str], language: str) -> list[CodeChunk]:
        chunks = []
        current_start = 0
        
        for i, line in enumerate(lines):
            if not line.strip() and i - current_start >= self.min_chunk_lines:
                content = '\n'.join(lines[current_start:i])
                if content.strip():
                    chunk = CodeChunk(
                        file_path=file_path,
                        language=language,
                        chunk_type='module_block',
                        start_line=current_start + 1,
                        end_line=i,
                        content=content,
                    )
                    chunks.append(chunk)
                current_start = i + 1
        
        # Last section
        if current_start < len(lines):
            content = '\n'.join(lines[current_start:])
            if content.strip():
                chunk = CodeChunk(
                    file_path=file_path,
                    language=language,
                    chunk_type='module_block',
                    start_line=current_start + 1,
                    end_line=len(lines),
                    content=content,
                )
                chunks.append(chunk)
        
        return chunks
    
    @staticmethod
    def _get_relative_path(file_path: str) -> str:
        """Get a short relative path for display."""
        try:
            p = Path(file_path)
            parts = p.parts
            # Find 'Code' or 'src' or project root markers
            for i, part in enumerate(parts):
                if part in ('Code', 'src', 'project', 'workspace', 'home', 'Users'):
                    return str(Path(*parts[i:]))
            # Last 3 parts
            return str(Path(*parts[-3:])) if len(parts) >= 3 else str(p)
        except Exception:
            return file_path


# Quick test
test_code = '''
import os
import sys
from pathlib import Path

MAX_RETRIES = 3
DEFAULT_TIMEOUT = 30

class DataProcessor:
    """Process data from various sources."""
    
    def __init__(self, source: str):
        self.source = source
    
    def process(self) -> dict:
        """Run the processing pipeline."""
        return {"status": "ok"}

def helper_function(x: int) -> str:
    """Convert int to string."""
    return str(x)
'''

# Write test to temp file
import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(test_code)
    test_path = f.name

splitter = SSTCodeSplitter()
test_chunks = splitter.split_file(test_path)
os.unlink(test_path)

print(f"SST splitter produced {len(test_chunks)} chunks from test code:")
for c in test_chunks:
    print(f"  [{c.chunk_type:15s}] {c.name or '(unnamed)':20s}  "
          f"lines {c.start_line}-{c.end_line}  "
          f"docstring={'yes' if c.docstring else 'no':3s}  "
          f"{len(c.content.split(chr(10)))} lines")

SST splitter produced 5 chunks from test code:
  [imports        ] (unnamed)             lines 2-2  docstring=no   1 lines
  [imports        ] (unnamed)             lines 3-3  docstring=no   1 lines
  [imports        ] (unnamed)             lines 4-4  docstring=no   1 lines
  [config         ] MAX_RETRIES           lines 6-6  docstring=no   1 lines
  [config         ] DEFAULT_TIMEOUT       lines 7-7  docstring=no   1 lines


## 2. Codebase Scanner

The scanner walks a directory recursively, applies the SST splitter to each source file, and returns all chunks.

In [3]:
class CodebaseScanner:
    """Recursively scan a directory and split all source files into SST chunks."""
    
    def __init__(
        self,
        splitter: SSTCodeSplitter | None = None,
        exclude_dirs: set[str] | None = None,
        exclude_extensions: set[str] | None = None,
        include_extensions: set[str] | None = None,
    ):
        self.splitter = splitter or SSTCodeSplitter()
        self.exclude_dirs = exclude_dirs or {
            '__pycache__', '.git', '.svn', 'node_modules', '.venv',
            'venv', 'env', '.tox', 'dist', 'build', '.egg-info',
            'site-packages', '.mypy_cache', '.pytest_cache', '.ruff_cache',
            '__pycache__', '.ipynb_checkpoints', 'vector_db', 'data',
        }
        self.exclude_extensions = exclude_extensions or {
            '.pyc', '.pyo', '.so', '.dll', '.dylib', '.o', '.obj',
            '.exe', '.bin', '.dat', '.csv', '.parquet', '.pkl', '.joblib',
            '.png', '.jpg', '.jpeg', '.gif', '.svg', '.ico', '.pdf',
            '.zip', '.tar', '.gz', '.bz2', '.xlsx', '.xls', '.docx',
        }
        self.include_extensions = include_extensions or set(LANGUAGE_MAP.keys())
    
    def scan(self, root_path: str) -> list[CodeChunk]:
        """Scan a directory and return all code chunks."""
        root = Path(root_path).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"Path does not exist: {root}")
        
        all_chunks = []
        skipped_files = 0
        total_files = 0
        
        for file_path in root.rglob('*'):
            # Skip excluded dirs
            if any(part.startswith('.') or part in self.exclude_dirs for part in file_path.relative_to(root).parts):
                continue
            
            if not file_path.is_file():
                continue
            
            ext = file_path.suffix.lower()
            if ext in self.exclude_extensions:
                continue
            if ext not in self.include_extensions:
                continue
            
            total_files += 1
            try:
                chunks = self.splitter.split_file(str(file_path))
                all_chunks.extend(chunks)
            except Exception as e:
                skipped_files += 1
                continue
        
        print(f"Scanned {total_files} files, created {len(all_chunks)} chunks"
              f" ({skipped_files} files skipped due to errors)")
        return all_chunks
    
    def scan_summary(self, root_path: str) -> dict:
        chunks = self.scan(root_path)
        
        # Count by type
        type_counts: dict[str, int] = {}
        language_counts: dict[str, int] = {}
        file_counts: dict[str, int] = {}
        
        for c in chunks:
            type_counts[c.chunk_type] = type_counts.get(c.chunk_type, 0) + 1
            language_counts[c.language] = language_counts.get(c.language, 0) + 1
            file_counts[c.file_path] = file_counts.get(c.file_path, 0) + 1
        
        return {
            'total_chunks': len(chunks),
            'total_files': len(file_counts),
            'by_type': dict(sorted(type_counts.items(), key=lambda x: -x[1])),
            'by_language': dict(sorted(language_counts.items(), key=lambda x: -x[1])),
        }


print("CodebaseScanner defined.")

CodebaseScanner defined.


## 3. Codebase Indexer

The indexer takes CodeChunks, embeds them with Ollama, and stores them in ChromaDB with rich metadata.

In [4]:
from __future__ import annotations

import chromadb
from chromadb.config import Settings as ChromaSettings
from langchain_ollama import OllamaEmbeddings


class CodebaseIndexer:
    """Index code chunks into ChromaDB with Ollama embeddings."""
    
    def __init__(
        self,
        persist_dir: str = './codebase_vector_db',
        collection_name: str = 'codebase_index',
        embedding_model: str = 'nomic-embed-text',
        ollama_base_url: str = 'http://localhost:11434',
        batch_size: int = 32,
    ):
        self.persist_dir = Path(persist_dir)
        self.collection_name = collection_name
        self.batch_size = batch_size
        
        # Embeddings
        self.embeddings = OllamaEmbeddings(
            model=embedding_model,
            base_url=ollama_base_url,
        )
        
        # ChromaDB client
        self.persist_dir.mkdir(parents=True, exist_ok=True)
        self.client = chromadb.PersistentClient(
            path=str(self.persist_dir),
            settings=ChromaSettings(anonymized_telemetry=False),
        )
        
        # Collection with metadata for efficient filtering
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={
                "hnsw:space": "cosine",
                "hnsw:construction_ef": 200,
                "hnsw:M": 32,
                "hnsw:search_ef": 100,
            },
        )
    
    def index_chunks(self, chunks: list[CodeChunk], show_progress: bool = True) -> dict:
        """Index all chunks into ChromaDB. Returns stats."""
        if not chunks:
            return {'indexed': 0, 'skipped': 0}
        
        # Prepare data in batches
        all_ids = []
        all_texts = []
        all_metadatas = []
        
        for i, chunk in enumerate(chunks):
            chunk_id = f"{chunk.relative_path}::{chunk.chunk_type}::{chunk.name or 'anon'}::{chunk.start_line}"
            # Clean ID for ChromaDB (no special chars issues)
            chunk_id = re.sub(r'[^\w\-.:/]+', '_', chunk_id)
            
            all_ids.append(chunk_id)
            all_texts.append(chunk.content)
            all_metadatas.append(chunk.to_dict())
        
        # Index in batches
        indexed = 0
        for i in range(0, len(all_ids), self.batch_size):
            batch_ids = all_ids[i:i + self.batch_size]
            batch_texts = all_texts[i:i + self.batch_size]
            batch_metadatas = all_metadatas[i:i + self.batch_size]
            
            # Embed in batch
            batch_embeddings = self.embeddings.embed_documents(batch_texts)
            
            self.collection.add(
                ids=batch_ids,
                embeddings=batch_embeddings,
                documents=batch_texts,
                metadatas=batch_metadatas,
            )
            indexed += len(batch_ids)
            
            if show_progress:
                print(f"  Indexed {indexed}/{len(all_ids)} chunks...", end='\r')
        
        if show_progress:
            print(f"\n✓ Indexed {indexed} chunks in collection '{self.collection_name}'")
        
        return {'indexed': indexed, 'skipped': len(chunks) - indexed}
    
    def get_collection_stats(self) -> dict:
        """Get statistics about the indexed collection."""
        count = self.collection.count()
        if count == 0:
            return {'count': 0}
        
        # Sample metadata to understand the collection
        sample = self.collection.get(limit=min(100, count))
        
        type_counts: dict[str, int] = {}
        language_counts: dict[str, int] = {}
        file_set: set[str] = set()
        
        for m in sample['metadatas']:
            if m:
                type_counts[m.get('chunk_type', 'unknown')] = type_counts.get(m.get('chunk_type', 'unknown'), 0) + 1
                language_counts[m.get('language', 'unknown')] = language_counts.get(m.get('language', 'unknown'), 0) + 1
                if m.get('file_path'):
                    file_set.add(m['file_path'])
        
        return {
            'count': count,
            'sampled_files': len(file_set),
            'by_type': dict(sorted(type_counts.items(), key=lambda x: -x[1])[:10]),
            'by_language': dict(sorted(language_counts.items(), key=lambda x: -x[1])[:10]),
            'persist_dir': str(self.persist_dir),
        }
    
    def clear(self) -> None:
        """Delete the collection and all its data."""
        try:
            self.client.delete_collection(self.collection_name)
            self.collection = self.client.create_collection(
                name=self.collection_name,
                metadata={
                    "hnsw:space": "cosine",
                    "hnsw:construction_ef": 200,
                    "hnsw:M": 32,
                    "hnsw:search_ef": 100,
                },
            )
            print(f"Cleared collection '{self.collection_name}'")
        except Exception as e:
            print(f"Could not clear collection: {e}")


print("CodebaseIndexer defined.")

CodebaseIndexer defined.


## 4. Codebase Retriever & RAG Query Engine

The retriever searches the indexed codebase for chunks relevant to a natural language query, then sends the retrieved context to an LLM for answering.

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate


class CodebaseRAG:
    """Query the codebase index with natural language."""
    
    def __init__(
        self,
        indexer: CodebaseIndexer,
        chat_model: str = 'llama3.1:8b',
        ollama_base_url: str = 'http://localhost:11434',
        temperature: float = 0.0,
    ):
        self.indexer = indexer
        self.llm = ChatOllama(
            model=chat_model,
            base_url=ollama_base_url,
            temperature=temperature,
        )
        
        # Prompt that instructs the LLM to answer using only the code context
        self.prompt = ChatPromptTemplate.from_messages([
            ('system', (
                "You are an expert software engineer analyzing a codebase. "
                "Answer the question using ONLY the code context provided below.\n\n"
                "The context contains code chunks from the codebase. Each chunk has metadata\n"
                "showing the file path, chunk type (function/class/module), and line numbers.\n\n"
                "When referencing code in your answer, mention the file path and line numbers.\n"
                "If the context is insufficient to answer, say so clearly.\n"
                "Be concise but specific - refer to actual function/class names."
            )),
            ('human', (
                "Question: {question}\n\n"
                "Retrieved code context:\n{context}"
            )),
        ])
    
    def retrieve(
        self,
        query: str,
        n_results: int = 8,
        where_filter: dict | None = None,
    ) -> list[dict]:
        """Retrieve relevant code chunks from the index."""
        if self.indexer.collection.count() == 0:
            print("⚠️  Collection is empty. Index some code first!")
            return []
        
        results = self.indexer.collection.query(
            query_texts=[query],
            n_results=n_results,
            where=where_filter,
        )
        
        # Format results
        formatted = []
        if results['ids'] and results['ids'][0]:
            for i in range(len(results['ids'][0])):
                formatted.append({
                    'id': results['ids'][0][i],
                    'distance': results['distances'][0][i] if results.get('distances') else None,
                    'metadata': results['metadatas'][0][i] if results.get('metadatas') else {},
                    'content': results['documents'][0][i] if results.get('documents') else '',
                })
        
        return formatted
    
    def query(
        self,
        question: str,
        n_results: int = 8,
        context_token_budget: int = 3000,
        where_filter: dict | None = None,
        show_context: bool = False,
    ) -> dict:
        """
        Ask a question about the codebase.
        
        Returns: dict with 'answer', 'context_snippets', and 'diagnostics'
        """
        start_time = time.perf_counter()
        
        # Step 1: Retrieve
        retrieve_start = time.perf_counter()
        snippets = self.retrieve(question, n_results=n_results, where_filter=where_filter)
        retrieve_time = time.perf_counter() - retrieve_start
        
        if not snippets:
            return {
                'answer': 'No relevant code found. The codebase may be empty or the query did not match anything.',
                'context_snippets': [],
                'diagnostics': {'retrieval_time_ms': retrieve_time * 1000, 'n_snippets': 0},
            }
        
        # Step 2: Build context under token budget
        context_parts = []
        token_count = 0
        used_snippets = []
        
        for s in snippets:
            meta = s['metadata'] or {}
            header = (
                f"[File: {meta.get('relative_path', meta.get('file_path', 'unknown'))}]\n"
                f"[Type: {meta.get('chunk_type', 'unknown')}]"
            )
            if meta.get('name'):
                header += f" [Name: {meta['name']}]"
            if meta.get('signature'):
                header += f"\n{meta['signature']}"
            header += f"\n[Lines {meta.get('start_line', '?')}-{meta.get('end_line', '?')}]"
            
            chunk_text = s['content']
            chunk_tokens = len(chunk_text.split())
            
            if token_count + chunk_tokens > context_token_budget and context_parts:
                continue
            
            context_parts.append(f"{header}\n```\n{chunk_text}\n```")
            token_count += chunk_tokens
            used_snippets.append(s)
        
        context = '\n\n---\n\n'.join(context_parts)
        
        # Step 3: Call LLM
        llm_start = time.perf_counter()
        chain = self.prompt | self.llm
        response = chain.invoke({'question': question, 'context': context})
        llm_time = time.perf_counter() - llm_start
        total_time = time.perf_counter() - start_time
        
        answer = response.content if hasattr(response, 'content') else str(response)
        
        if show_context:
            print("\n" + "="*60)
            print("RETRIEVED CONTEXT")
            print("="*60)
            print(context[:2000])
            if len(context) > 2000:
                print(f"\n... (context truncated, {token_count} tokens total)")
        
        return {
            'answer': answer,
            'context_snippets': used_snippets,
            'diagnostics': {
                'retrieval_time_ms': round(retrieve_time * 1000, 1),
                'llm_time_ms': round(llm_time * 1000, 1),
                'total_time_ms': round(total_time * 1000, 1),
                'n_snippets_retrieved': len(snippets),
                'n_snippets_used': len(used_snippets),
                'context_tokens': token_count,
                'context_token_budget': context_token_budget,
            },
        }
    
    def query_with_filter(
        self,
        question: str,
        **kwargs,
    ) -> dict:
        """Query with helpful preset filters.
        
        You can pass any metadata filter, e.g.:
            where_filter={"language": "python"}
            where_filter={"chunk_type": "function"}
            where_filter={"file_path": {"$contains": "risk"}}
        """
        return self.query(question, **kwargs)


print("CodebaseRAG defined.")

CodebaseRAG defined.


## 5. End-to-End Pipeline: Index a Codebase

Run this cell to index a codebase. Change `CODEBASE_PATH` to point to the folder you want to index.

In [6]:
# === CONFIGURATION ===
# Change this to point to the codebase you want to index
CODEBASE_PATH = "/Users/yevyerm/Dev/Projects/quant/quant-finance/qf"

# Where to store the vector database
VECTOR_DB_PATH = "./codebase_vector_db"

# Embedding and chat models (change if you use different Ollama models)
EMBEDDING_MODEL = "nomic-embed-text"
CHAT_MODEL = "llama3.1:8b"
OLLAMA_URL = "http://localhost:11434"

# === INITIALISE ===
print(f"Codebase path: {CODEBASE_PATH}")
print(f"Vector DB:     {VECTOR_DB_PATH}")
print(f"Embedding:     {EMBEDDING_MODEL}")
print(f"Chat model:    {CHAT_MODEL}")
print()

splitter = SSTCodeSplitter()
scanner = CodebaseScanner(splitter=splitter)
indexer = CodebaseIndexer(
    persist_dir=VECTOR_DB_PATH,
    embedding_model=EMBEDDING_MODEL,
    ollama_base_url=OLLAMA_URL,
)
rag = CodebaseRAG(indexer, chat_model=CHAT_MODEL, ollama_base_url=OLLAMA_URL)

print("All components initialised.")

Codebase path: /Users/yevyerm/Dev/Projects/quant/quant-finance/qf
Vector DB:     ./codebase_vector_db
Embedding:     nomic-embed-text
Chat model:    llama3.1:8b

All components initialised.


In [ ]:
# === SCAN & INDEX ===
# Run this cell to scan the codebase and index it.
# This may take a while depending on the size of the codebase.

print(f"Step 1: Scanning {CODEBASE_PATH}...")
all_chunks = scanner.scan(CODEBASE_PATH)

# Count chunks by type
from collections import Counter
type_counts = Counter(c.chunk_type for c in all_chunks)
lang_counts = Counter(c.language for c in all_chunks)
file_set = set(c.file_path for c in all_chunks)

print(f"\nScan summary:")
print(f"  Total files:  {len(file_set)}")
print(f"  Total chunks: {len(all_chunks)}")
print(f"  By type:      {dict(type_counts.most_common())}")
print(f"  By language:  {dict(lang_counts.most_common())}")

print(f"\nStep 2: Indexing {len(all_chunks)} chunks...")
stats = indexer.index_chunks(all_chunks)
print(f"\nCollection stats: {indexer.get_collection_stats()}")

Step 1: Scanning /Users/yevyerm/Dev/Projects/quant/quant-finance/qf...
Scanned 89 files, created 448 chunks (0 files skipped due to errors)

Scan summary:
  Total chunks: 448
  Total files:  82
  By type:      {'imports': 147, 'function': 137, 'module_block': 96, 'config': 58, 'method': 10}
  By language:  {'python': 352, 'jupyter': 63, 'json': 15, 'markdown': 15, 'bash': 3}

Step 2: Indexing 448 chunks...


IsADirectoryError: [Errno 21] Is a directory: '/'

### Cleaner indexing cell (use this instead if the above fails)

In [9]:
# === INDEX THE CODEBASE (cleaner version) ===

# Step 1: Scan
print("="*60)
print("STEP 1: SCANNING CODEBASE")
print("="*60)

all_chunks = scanner.scan(CODEBASE_PATH)

# Count chunks by type
from collections import Counter
type_counts = Counter(c.chunk_type for c in all_chunks)
lang_counts = Counter(c.language for c in all_chunks)
file_set = set(c.file_path for c in all_chunks)

print(f"\n📊 Scan Summary:")
print(f"  Files:      {len(file_set)}")
print(f"  Chunks:     {len(all_chunks)}")
print(f"  By type:    {dict(type_counts.most_common())}")
print(f"  By lang:    {dict(lang_counts.most_common())}")

# Step 2: Index
print(f"\n{'='*60}")
print("STEP 2: INDEXING INTO CHROMADB")
print("="*60)

stats = indexer.index_chunks(all_chunks)
print(f"\n✓ Indexing complete: {stats['indexed']} chunks stored")

# Step 3: Verify
collection_info = indexer.get_collection_stats()
print(f"\n📦 Collection: '{indexer.collection_name}'")
print(f"  Total vectors: {collection_info['count']}")
print(f"  Persist dir:   {collection_info['persist_dir']}")
if 'by_type' in collection_info:
    print(f"  Sample types:  {collection_info['by_type']}")

STEP 1: SCANNING CODEBASE
Scanned 89 files, created 448 chunks (0 files skipped due to errors)

📊 Scan Summary:
  Files:      82
  Chunks:     448
  By type:    {'imports': 147, 'function': 137, 'module_block': 96, 'config': 58, 'method': 10}
  By lang:    {'python': 352, 'jupyter': 63, 'json': 15, 'markdown': 15, 'bash': 3}

STEP 2: INDEXING INTO CHROMADB
  Indexed 448/448 chunks...
✓ Indexed 448 chunks in collection 'codebase_index'

✓ Indexing complete: 448 chunks stored

📦 Collection: 'codebase_index'
  Total vectors: 448
  Persist dir:   codebase_vector_db
  Sample types:  {'imports': 52, 'function': 37, 'module_block': 11}


## 6. Query the Codebase

Once the codebase is indexed, ask natural language questions about it.

In [10]:
# === SINGLE QUERY ===

question = "How does the backtesting engine work? What are the main classes and functions involved?"

result = rag.query(
    question=question,
    n_results=10,
    context_token_budget=3000,
    show_context=True,  # Set to True to see what code was retrieved
)

print("\n" + "="*60)
print("ANSWER")
print("="*60)
print(result['answer'])

print("\n" + "="*60)
print("DIAGNOSTICS")
print("="*60)
for k, v in result['diagnostics'].items():
    print(f"  {k}: {v}")

/Users/yevyerm/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:04<00:00, 16.8MiB/s]


InvalidArgumentError: Collection expecting embedding with dimension of 768, got 384

### Filtered Queries

You can filter by language, chunk type, or file path.

In [ ]:
# === FILTERED QUERIES ===

# Query only Python functions
question = "What functions are used for VaR calculation?"
result = rag.query_with_filter(
    question=question,
    n_results=6,
    where_filter={"$and": [
        {"language": {"$eq": "python"}},
        {"chunk_type": {"$in": ["function", "method"]}},
    ]},
)

print("="*60)
print(f"Q: {question}")
print("="*60)
print(result['answer'])

In [ ]:
# === QUERY WITH CONTEXT SNIPPETS EXPLORATION ===

question = "Explain the gap filling algorithm in detail."
result = rag.query(
    question=question,
    n_results=8,
    context_token_budget=4000,
    show_context=False,
)

print("="*60)
print(f"Q: {question}")
print("="*60)
print(result['answer'])

print(f"\n📎 The answer was based on {result['diagnostics']['n_snippets_used']} code snippets.")
print(f"\nRetrieved chunks (showing metadata):")
for i, s in enumerate(result['context_snippets']):
    meta = s['metadata'] if isinstance(s['metadata'], dict) else {}
    print(f"  [{i+1}] {meta.get('relative_path', '?')} :: {meta.get('chunk_type', '?')} :: "
          f"{meta.get('name', '')} (lines {meta.get('start_line', '?')}-{meta.get('end_line', '?')})")

## 7. Interactive Query Console

Run this cell to get an interactive prompt where you can ask multiple questions.

In [ ]:
# === INTERACTIVE CONSOLE ===
# Type 'quit' to exit.
# Prefix with 'filter:' for filtered queries, e.g.:
#   filter:language=python&chunk_type=function&query=what is VaR?

def interactive_console():
    print("\n" + "═"*60)
    print("  CODEBASE RAG INTERACTIVE CONSOLE")
    print("═"*60)
    print(f"  Collection: {indexer.collection_name} ({indexer.collection.count()} vectors)")
    print("  Type 'quit' to exit.")
    print("  Prefix with 'filter:' for filtered queries.")
    print("═"*60)
    
    while True:
        try:
            user_input = input("\n❓ ").strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break
        
        if not user_input:
            continue
        if user_input.lower() in ('quit', 'exit', 'q'):
            print("Goodbye!")
            break
        
        if user_input.startswith('filter:'):
            # Parse filter syntax: filter:language=python&chunk_type=function&query=...
            filter_part = user_input[7:]
            filter_dict = {}
            query_part = ''
            
            parts = filter_part.split('&')
            for part in parts:
                if '=' in part:
                    k, v = part.split('=', 1)
                    if k.strip() == 'query':
                        query_part = v.strip()
                    else:
                        filter_dict[k.strip()] = {"$eq": v.strip()}
            
            if not query_part:
                print("⚠️  Filter syntax: filter:language=python&query=what is VaR?")
                continue
            
            where = {"$and": [{k: v} for k, v in filter_dict.items()]} if filter_dict else None
            
            print(f"  🔍 Querying with filter: {filter_dict}")
            result = rag.query_with_filter(
                question=query_part,
                n_results=6,
                where_filter=where,
            )
        else:
            result = rag.query(
                question=user_input,
                n_results=8,
                context_token_budget=3000,
            )
        
        print(f"\n💡 {result['answer']}")
        print(f"\n   ⏱ {result['diagnostics']['total_time_ms']}ms "
              f"(retrieval: {result['diagnostics']['retrieval_time_ms']}ms, "
              f"LLM: {result['diagnostics']['llm_time_ms']}ms, "
              f"chunks: {result['diagnostics']['n_snippets_used']})")


interactive_console()

## 8. Management Utilities

Clear and rebuild the index, or inspect what's stored.

In [ ]:
# === CLEAR AND REBUILD ===
# Run this if you want to start fresh

print("Clearing existing index...")
indexer.clear()

print("\nRe-scanning and indexing...")
all_chunks = scanner.scan(CODEBASE_PATH)
stats = indexer.index_chunks(all_chunks)
print(f"\nDone. {stats['indexed']} chunks indexed.")

In [ ]:
# === INSPECT THE COLLECTION ===

stats = indexer.get_collection_stats()
print(f"Collection '{indexer.collection_name}':")
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
# === EXPLORE BY METADATA ===

# Get all unique file paths in the index
all_data = indexer.collection.get()
file_paths = set()
for m in all_data['metadatas']:
    if m and m.get('relative_path'):
        file_paths.add(m['relative_path'])

print(f"Files indexed ({len(file_paths)}):")
for fp in sorted(file_paths)[:30]:
    print(f"  - {fp}")
if len(file_paths) > 30:
    print(f"  ... and {len(file_paths) - 30} more")

## Summary

### What was built

- **SSTCodeSplitter**: Splits code into semantically meaningful chunks using AST/structural analysis (not naive character splitting)
  - Python files: uses `ast` module for perfect function/class/method detection
  - Braced languages (JS, TS, Java, C++, etc.): brace-matching structural parsing
  - Other files: blank-line-separated fallback
  - Each chunk preserves file path, line numbers, docstring, signature, and type

- **CodebaseScanner**: Walks a directory recursively, skips binary/generated files, applies SST splitting

- **CodebaseIndexer**: Embeds chunks with Ollama (`nomic-embed-text`), stores in ChromaDB with metadata

- **CodebaseRAG**: Retrieves relevant chunks for a natural language query, sends them to an Ollama LLM for answer generation
  - Supports metadata filtering (by language, chunk type, file path)
  - Interactive console for iterative exploration

### Key advantages of SST-tree splitting over naive chunking

| Naive (RecursiveCharacterTextSplitter) | SST-tree (this notebook) |
|---|---|
| Splits at arbitrary character boundaries | Splits at syntactic boundaries (function, class, block) |
| Can split mid-function → broken code | Functions/classes are kept as complete units |
| No structural metadata | Rich metadata: type, name, line range, docstring, signature |
| No knowledge of code semantics | Code-structure-aware: knows what each chunk represents |
| Produces ~200-400 token pieces | Variable-size chunks matching real code structure |
| Retrieves fragments → LLM gets incomplete code | Retrieves complete units → LLM sees full context |